# DwarfsRafting

[Codility task](https://app.codility.com/programmers/trainings/2/dwarfs_rafting/) - Exercise 2, 2016 Contest

A square raft has `N x N` seats (rows `1..N`, columns `A, B, C, ...`). `S` lists the seats with barrels and `T` the seats already taken by dwarfs. The dwarfs may use the remaining seats, but the front and back halves (rows) must hold the same number of dwarfs, and so must the left and right halves (columns). `solution(N, S, T)` returns the maximum number of **additional** dwarfs, or `-1` if the raft cannot be balanced.

| N | S | T | result |
|---|---|---|--------|
| 4 | `"1B 1C 4B 1D 2A"` | `"3B 2D"` | 6 |

Assumption: `N` is even, in `[2..26]`.

## Idea

Split the raft into four quadrants: front-left (FL), front-right (FR), back-left (BL), back-right (BR). The two conditions

- `FL + FR = BL + BR` (front = back),
- `FL + BL = FR + BR` (left = right),

are together equivalent to `FL = BR` **and** `FR = BL` (add and subtract them).

For each quadrant let `t` be the dwarfs already seated and `cap` the seats that are not barrels (which includes the seated dwarfs). The common value `p` of two opposite quadrants must satisfy `max(t1, t2) <= p <= min(cap1, cap2)`: if that range is empty the answer is `-1`, otherwise take `p = min(cap1, cap2)`. The total is `2 p_1 + 2 p_2`, and the answer counts only the **new** dwarfs: subtract the dwarfs in `T`.

In [ ]:
def parse(seats):
    """'12C 3A' -> [(12, 2), (3, 0)]: 1-based row, 0-based column."""
    return [(int(seat[:-1]), ord(seat[-1]) - ord("A")) for seat in seats.split()]


def solution(N, S, T):
    half = N // 2
    barrels = [[0, 0], [0, 0]]                       # indexed [front/back][left/right]
    dwarfs = [[0, 0], [0, 0]]
    for row, col in parse(S):
        barrels[(row - 1) // half][col // half] += 1
    seated = parse(T)
    for row, col in seated:
        dwarfs[(row - 1) // half][col // half] += 1
    capacity = [[half * half - barrels[r][c] for c in range(2)] for r in range(2)]

    total = 0
    for (r1, c1), (r2, c2) in (((0, 0), (1, 1)), ((0, 1), (1, 0))):      # opposite quadrants
        low = max(dwarfs[r1][c1], dwarfs[r2][c2])
        high = min(capacity[r1][c1], capacity[r2][c2])
        if low > high:
            return -1
        total += 2 * high
    return total - len(seated)

## Official example

In [ ]:
assert solution(4, "1B 1C 4B 1D 2A", "3B 2D") == 6
print("example OK")

## Edge cases

In [ ]:
assert solution(2, "", "") == 4                          # empty raft: everybody fits
assert solution(2, "1A", "") == 2                        # FL is gone, so BR must be empty; FR and BL stay
assert solution(2, "1A 2B", "") == 2                    # FL and BR are barrels: only FR and BL (2 seats) are used
assert solution(2, "", "1A 2B") == 2                     # dwarfs in opposite quadrants: FR and BL fill up
assert solution(2, "", "1A") == 3                        # 1A seated: BR must also have a dwarf, and FR and BL
assert solution(2, "2B", "1A") == -1                     # FL has a dwarf but BR is a barrel: impossible
assert solution(2, "", "1A 1B") == 2                     # two dwarfs in the front row: FL = BR = 1 and FR = BL = 1
assert solution(2, "1A 1B 2A 2B", "") == 0               # nothing but barrels
assert solution(26, "", "") == 26 * 26
assert solution(26, "26Z", "") == 26 * 26 - 2            # a barrel in BR caps FL and BR at 168: two seats are lost
print("edge cases OK")

## Exhaustive check on `N = 2` and literal brute force on `N = 4`

The reference follows the statement to the letter: it enumerates **every subset** of the free seats, checks that the front and back rows and the left and right columns hold the same number of dwarfs, and keeps the largest number of new dwarfs (or `-1`).

In [ ]:
import itertools
import random


def seat_name(row, col):
    return f"{row}{chr(ord('A') + col)}"


def brute(N, S, T):
    half = N // 2
    barrels = set(parse(S))
    seated = set(parse(T))
    free = [(r, c) for r in range(1, N + 1) for c in range(N) if (r, c) not in barrels and (r, c) not in seated]
    best = -1
    for mask in range(1 << len(free)):
        chosen = [free[i] for i in range(len(free)) if mask >> i & 1]
        dwarfs = list(seated) + chosen
        front = sum(1 for r, c in dwarfs if r <= half)
        left = sum(1 for r, c in dwarfs if c < half)
        if front == len(dwarfs) - front and left == len(dwarfs) - left:
            best = max(best, len(chosen))
    return best


# N = 2: every seat is free, a barrel or a dwarf (3^4 = 81 configurations)
checked = 0
for states in itertools.product("fbd", repeat=4):
    seats = [seat_name(r, c) for r in (1, 2) for c in (0, 1)]
    S = " ".join(s for s, st in zip(seats, states) if st == "b")
    T = " ".join(s for s, st in zip(seats, states) if st == "d")
    assert solution(2, S, T) == brute(2, S, T), (S, T)
    checked += 1
print(f"N = 2: {checked} configurations agree")

# N = 4: random configurations with at most 12 free seats (so that subsets can be enumerated)
rng = random.Random(8)
cells = [seat_name(r, c) for r in range(1, 5) for c in range(4)]
count = 0
for _ in range(400):
    k = rng.randint(4, 12)
    chosen = rng.sample(cells, k)
    S = " ".join(rng.sample(chosen, rng.randint(0, k)))
    T = " ".join(c for c in chosen if c not in S.split())
    assert solution(4, S, T) == brute(4, S, T), (S, T)
    count += 1
print(f"N = 4: {count} random configurations agree")

## Quadrant brute force for bigger rafts

For `N` up to 8 each quadrant has at most 16 seats, so the number of dwarfs to add to each quadrant can be enumerated (4 nested ranges). The reference checks the two balance conditions on the four totals, exactly as written in the statement.

In [ ]:
def brute_quadrants(N, S, T):
    half = N // 2
    barrels = [[0, 0], [0, 0]]
    dwarfs = [[0, 0], [0, 0]]
    for row, col in parse(S):
        barrels[(row - 1) // half][col // half] += 1
    for row, col in parse(T):
        dwarfs[(row - 1) // half][col // half] += 1
    free = [half * half - barrels[r][c] - dwarfs[r][c] for r in range(2) for c in range(2)]
    base = [dwarfs[r][c] for r in range(2) for c in range(2)]
    best = -1
    for add in itertools.product(*(range(f + 1) for f in free)):
        fl, fr, bl, br = (b + a for b, a in zip(base, add))
        if fl + fr == bl + br and fl + bl == fr + br:
            best = max(best, sum(add))
    return best


all_seats = {n: [seat_name(r, c) for r in range(1, n + 1) for c in range(n)] for n in (2, 4, 6, 8)}
for _ in range(600):
    N = rng.choice([2, 4, 6, 8])
    seats = all_seats[N]
    k = rng.randint(N * N // 4, N * N // 2)
    chosen = rng.sample(seats, k)
    split = rng.randint(0, k)
    S, T = " ".join(chosen[:split]), " ".join(chosen[split:])
    assert solution(N, S, T) == brute_quadrants(N, S, T), (N, S, T)
print("matches the quadrant enumeration on 600 random rafts (N up to 8)")

## Performance (N = 26)

In [ ]:
import time

seats = [seat_name(r, c) for r in range(1, 27) for c in range(26)]
chosen = rng.sample(seats, 300)
S, T = " ".join(chosen[:150]), " ".join(chosen[150:])
start = time.perf_counter()
result = solution(26, S, T)
print(f"N = 26 with 300 occupied seats -> {result} in {time.perf_counter() - start:.5f}s")

## Notes

- The two conditions look coupled, but adding and subtracting them splits them into `FL = BR` and `FR = BL`: independent problems for two pairs of quadrants.
- `cap` includes the seated dwarfs, so the interval `[max t, min cap]` encodes both "cannot remove dwarfs" and "cannot use barrels".
- The function returns **new** dwarfs; do not forget to subtract the seated ones.
- Parsing: row numbers can have two digits (`"26Z"`), the column is the last character.